# Medication Recommender

In [ ]:
from pathlib import Path

print('current directory:', Path.cwd())

## Prepare

In [ ]:
# this file is to prepare the dataset
# take raw tables and turn them into a format for modeling
# we will end up with 2 tables
# one is (patient, admission) -> patient features, admission features, history
# the other one is (patient, admission, drug) -> label

from pathlib import Path
import json

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, save_npz

DATA_DIR = Path("data/baseline_tables")
OUT_DIR = Path("data/processed")

RANDOM_STATE = 42

# negative sampling
# model needs both positive and negative examples to learn what is good/bad
# we can change this to per positive.
NEGATIVES_PER_ADMISSION = 10

In [ ]:
# build diagnoses matrix
# use multi-hot encoding matrix
# returns:
# a sparse matrix (num_admissions, num_icd_codes)
# and a list of ICD codes, mapping column index to code
def build_current_dx_matrix(snapshot, diagnoses):
    # map each hadm_id to row number
    snap_hadms = snapshot["hadm_id"].tolist()
    hadm_to_row = {h: i for i, h in enumerate(snap_hadms)}

    dx = diagnoses.dropna(subset=["hadm_id", "icd_code"]).copy()
    dx["icd_code"] = dx["icd_code"].fillna("").astype(str).str.strip()
    dx = dx[dx["icd_code"] != ""]
    dx = dx[dx["hadm_id"].isin(hadm_to_row)]
    dx = dx[["hadm_id", "icd_code"]].drop_duplicates()

    dx_codes = sorted(dx["icd_code"].unique())
    # map icd code to column number
    code_to_col = {code: i for i, code in enumerate(dx_codes)}

    # now build sparse matrix (multi-hot)
    rows = dx["hadm_id"].map(hadm_to_row).values
    cols = dx["icd_code"].map(code_to_col).values
    data = np.ones(len(dx), dtype=np.int8)
    matrix = csr_matrix((data, (rows, cols)), shape=(snapshot.shape[0], len(dx_codes)))
    return matrix, dx_codes

In [ ]:
# HISTORY (patient's past admissions)
# includes prior diagnoses, medications, number of admissions, days since last admission
# basically collapsing user history into a fixed size feature vector
def compute_history(snapshot, diagnoses, interactions, dx_codes, medications):
    dx_to_col = {c: i for i, c in enumerate(dx_codes)}
    med_to_col = {m: i for i, m in enumerate(medications)}

    n_rows = len(snapshot)
    num_prior_admissions = np.zeros(n_rows, dtype=int)
    days_since_last_admission = np.full(n_rows, np.nan)

    # per admission diagnosis/medication sets
    dx_clean = diagnoses.dropna(subset=["hadm_id", "icd_code"]).copy()
    dx_clean["icd_code"] = dx_clean["icd_code"].fillna("").astype(str).str.strip()
    dx_clean = dx_clean[dx_clean["icd_code"] != ""]

    current_dx_by_hadm = {}
    for hadm_id, grp in dx_clean.groupby("hadm_id"):
        current_dx_by_hadm[hadm_id] = set(grp["icd_code"])

    current_meds_by_hadm = {}
    for hadm_id, grp in interactions.groupby("hadm_id"):
        current_meds_by_hadm[hadm_id] = set(grp["medication"])

    prior_dx_rows, prior_dx_cols = [], []
    prior_med_rows, prior_med_cols = [], []

    # go through each patient's admissions in chronological order
    # and build their history
    for _, group in snapshot.groupby("subject_id", sort=False):
        seen_dx = set()
        seen_meds = set()
        last_admit_time = None
        prior_count = 0

        for row in group.itertuples():
            i = row.Index
            num_prior_admissions[i] = prior_count

            # days_since_last_admission
            if last_admit_time is not None:
                gap = (row.admittime - last_admit_time).days
                days_since_last_admission[i] = gap

            for code in seen_dx:
                if code in dx_to_col:
                    prior_dx_rows.append(i)
                    prior_dx_cols.append(dx_to_col[code])
            for med in seen_meds:
                if med in med_to_col:
                    prior_med_rows.append(i)
                    prior_med_cols.append(med_to_col[med])

            if row.hadm_id in current_dx_by_hadm:
                seen_dx.update(current_dx_by_hadm[row.hadm_id])
            if row.hadm_id in current_meds_by_hadm:
                seen_meds.update(current_meds_by_hadm[row.hadm_id])
            last_admit_time = row.admittime
            prior_count += 1

    dx_data = np.ones(len(prior_dx_rows), dtype=np.int8)
    prior_dx_matrix = csr_matrix(
        (dx_data, (prior_dx_rows, prior_dx_cols)),
        shape=(n_rows, len(dx_codes)),
    )

    med_data = np.ones(len(prior_med_rows), dtype=np.int8)
    prior_med_matrix = csr_matrix(
        (med_data, (prior_med_rows, prior_med_cols)),
        shape=(n_rows, len(medications)),
    )

    history = pd.DataFrame(
        {
            "num_prior_admissions": num_prior_admissions,
            "days_since_last_admission": days_since_last_admission,
        },
        index=snapshot.index,
    )
    return history, prior_dx_matrix, prior_med_matrix

In [ ]:
# build the label table for admission drug pairs (final interaction table)
# build negative samples for training
def make_label_table(snapshot, interactions):
    rng = np.random.default_rng(RANDOM_STATE)
    all_drugs = sorted(interactions["medication"].unique())

    positive_by_hadm = {}
    for hadm_id, grp in interactions.groupby("hadm_id"):
        positive_by_hadm[hadm_id] = set(grp["medication"])

    rows = []
    for hadm_id in snapshot["hadm_id"]:
        positive_drugs = positive_by_hadm.get(hadm_id, set())

        for drug in sorted(positive_drugs):
            rows.append({"hadm_id": hadm_id, "candidate_drug": drug, "label": 1})

        # sample negatives from drugs not given in this admission
        non_positive_drugs = [d for d in all_drugs if d not in positive_drugs]

        n_neg = min(NEGATIVES_PER_ADMISSION, len(non_positive_drugs))
        negative_drugs = rng.choice(non_positive_drugs, size=n_neg, replace=False)

        for drug in sorted(negative_drugs):
            rows.append({"hadm_id": hadm_id, "candidate_drug": drug, "label": 0})

    return pd.DataFrame(rows)

In [ ]:
# === put everything together ===

print("loading raw tables...")
admissions = pd.read_csv(DATA_DIR / "admissions.csv", low_memory=False)
patients = pd.read_csv(DATA_DIR / "patients.csv", low_memory=False)
diagnoses = pd.read_csv(DATA_DIR / "diagnoses_icd.csv", low_memory=False)
emar = pd.read_csv(DATA_DIR / "emar.csv", low_memory=False)
print("admissions:", len(admissions), "patients:", len(patients))
print("diagnoses:", len(diagnoses), "emar:", len(emar))


# creates (hadm_id, medication) foundation
# no matching hadm_id - means not tied to an admission
interactions = emar.dropna(subset=["hadm_id"]).copy()
interactions["hadm_id"] = interactions["hadm_id"].astype(int)

meds = interactions["medication"].fillna("")
meds = meds.astype(str).str.strip()
interactions["medication"] = meds
interactions = interactions[interactions["medication"] != ""]

# for each admission, what drugs were administered (no dups)
interactions = interactions[["hadm_id", "medication"]].drop_duplicates()
interactions = interactions.reset_index(drop=True)
print("interactions:", len(interactions))


# start by joining patients/admissions, building basic blocks
# one row per admission
# patient is an easy merge into admissions
snapshot = admissions.merge(patients, on="subject_id", how="left")
snapshot["admittime"] = pd.to_datetime(snapshot["admittime"])

# get age, we need to calculate this because of how MIMIC shuffles data
snapshot["age_at_admission"] = (
    snapshot["anchor_age"] + snapshot["admittime"].dt.year - snapshot["anchor_year"]
)
snapshot = snapshot.drop(columns=["anchor_age", "anchor_year"])

# remove admissions that don't have any medication records (useless to us)
valid_hadms = set(interactions["hadm_id"])
snapshot = snapshot[snapshot["hadm_id"].isin(valid_hadms)]

# sort so we can create user history later.
snapshot = snapshot.sort_values(["subject_id", "admittime", "hadm_id"])
snapshot = snapshot.reset_index(drop=True)
print("snapshot:", snapshot.shape)


print("building current dx matrix...")
current_dx_matrix, dx_codes = build_current_dx_matrix(snapshot, diagnoses)
medications = sorted(interactions["medication"].unique())
print("dx codes:", len(dx_codes), "medications:", len(medications))

print("computing history features...")
history_features, prior_dx_matrix, prior_med_matrix = compute_history(
    snapshot, diagnoses, interactions, dx_codes, medications
)

patient_admission_snapshot = pd.concat([snapshot, history_features], axis=1)

print("making label table...")
admission_drug_labels = make_label_table(snapshot, interactions)
print("label rows:", len(admission_drug_labels))


# save everything

OUT_DIR.mkdir(parents=True, exist_ok=True)
patient_admission_snapshot.to_csv(
    OUT_DIR / "patient_admission_snapshot.csv", index=False
)
admission_drug_labels.to_csv(OUT_DIR / "admission_drug_labels.csv", index=False)
save_npz(OUT_DIR / "current_dx_matrix.npz", current_dx_matrix)
save_npz(OUT_DIR / "prior_dx_matrix.npz", prior_dx_matrix)
save_npz(OUT_DIR / "prior_med_matrix.npz", prior_med_matrix)

metadata = {
    "dx_codes": dx_codes,
    "medications": medications,
    "negatives_per_admission": NEGATIVES_PER_ADMISSION,
    "random_state": RANDOM_STATE,
}
out_path = OUT_DIR / "feature_metadata.json"
fp = open(out_path, "w")
json.dump(metadata, fp, indent=2)
fp.close()

print("done")

# final files:
# - patient_admission_snapshot.csv
# - admission_drug_labels.csv
# - current_dx_matrix.npz
# - prior_dx_matrix.npz
# - prior_med_matrix.npz
# - feature_metadata.json

## EDA

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import load_npz

DATA_DIR = Path("data/baseline_tables")
PROCESSED_DIR = Path("data/processed")
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)


# === DATA CHECKS SECTION ===

print("=" * 60)
print("DATA CHECKS")
print("=" * 60)

snapshot_path = PROCESSED_DIR / "patient_admission_snapshot.csv"
labels_path = PROCESSED_DIR / "admission_drug_labels.csv"
matrix_files = {
    "current_dx": PROCESSED_DIR / "current_dx_matrix.npz",
    "prior_dx": PROCESSED_DIR / "prior_dx_matrix.npz",
    "prior_med": PROCESSED_DIR / "prior_med_matrix.npz",
}

# run prepare.py before running

# load feature table
snapshot = pd.read_csv(snapshot_path, low_memory=False)
n_snap = len(snapshot)
print("snapshot loaded: %d rows" % n_snap)

# check encoded features
missing_matrices = []
for name, path in matrix_files.items():
    if not path.exists():
        missing_matrices.append(name)

if missing_matrices:
    print(f"missing matrices: {missing_matrices}")
else:
    # all of these should have same # of rows
    current_dx = load_npz(matrix_files["current_dx"])
    prior_dx = load_npz(matrix_files["prior_dx"])
    prior_med = load_npz(matrix_files["prior_med"])
    row_counts_match = (
        n_snap == current_dx.shape[0]
        and n_snap == prior_dx.shape[0]
        and n_snap == prior_med.shape[0]
    )
    print(f"snapshot/matrix row counts match: {row_counts_match}")

# interactions table
if not labels_path.exists():
    print(f"missing {labels_path}")
else:
    labels = pd.read_csv(labels_path, low_memory=False)
    print(f"labels loaded: {len(labels):,} rows")
    # check hasm_id match in both tables
    hadm_ids_match = labels["hadm_id"].isin(snapshot["hadm_id"]).all()
    # check dup (hadm_id, candidate_drug) pairs
    duplicate_pairs = labels.duplicated(["hadm_id", "candidate_drug"]).sum()
    # is our negative sampling good
    pair_label_counts = labels.groupby(["hadm_id", "candidate_drug"])["label"].nunique()
    conflict_count = (pair_label_counts > 1).sum()
    print(f"all label hadm_id values exist in snapshot: {hadm_ids_match}")
    print(f"duplicate (hadm_id, candidate_drug) rows: {duplicate_pairs:,}")
    print(f"admission-drug label conflicts: {conflict_count:,}")

In [ ]:
# === RAW TABLES SECTION ===

print("=" * 60)
print("RAW TABLES")
print("=" * 60)

# these are raw tables, before processing
admissions = pd.read_csv(DATA_DIR / "admissions.csv", low_memory=False)
patients = pd.read_csv(DATA_DIR / "patients.csv", low_memory=False)
diagnoses = pd.read_csv(DATA_DIR / "diagnoses_icd.csv", low_memory=False)
emar = pd.read_csv(DATA_DIR / "emar.csv", low_memory=False)

# row counts
print(f"admissions: {len(admissions):,} rows")
print(f"patients: {len(patients):,} rows")
print(f"diagnoses: {len(diagnoses):,} rows")
print(f"emar: {len(emar):,} rows")

# unique ids
print(f"unique patients in admissions: {admissions['subject_id'].nunique():,}")
# how many admissions tied to emar medications
print(f"unique admissions in emar: {emar['hadm_id'].nunique():,}")

# admissions per patient (cold start)
admissions_per_patient = admissions.groupby("subject_id").size()
print("\nadmissions per patient:")
desc = admissions_per_patient.describe().round(2)
print(desc.to_string())
single_visit = (admissions_per_patient == 1).sum()
multi_visit = (admissions_per_patient > 1).sum()
print(f"single-visit patients: {single_visit:,}")
print(f"multi-visit patients: {multi_visit:,}")

# diagnoses per admission
dx_per_admission_raw = diagnoses.groupby("hadm_id").size()
print("\ndiagnoses per admission (raw):")
desc = dx_per_admission_raw.describe().round(2)
print(desc.to_string())

# medication per admission
# no medication per admission, because of dups (since raw tables)
emar_per_admission = emar.groupby("hadm_id").size()
print("\nemar rows per admission:")
desc = emar_per_admission.describe().round(2)
print(desc.to_string())

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(admissions_per_patient, bins=50, edgecolor="white")
axes[0].set_title("admissions per patient")
axes[0].set_xlabel("admissions")
axes[0].set_ylabel("patients")
axes[0].set_yscale("log")

axes[1].hist(dx_per_admission_raw, bins=50, edgecolor="white")
axes[1].set_title("diagnoses per admission")
axes[1].set_xlabel("diagnoses")
axes[1].set_ylabel("admissions")

axes[2].hist(emar_per_admission, bins=50, edgecolor="white")
axes[2].set_title("emar rows per admission")
axes[2].set_xlabel("rows")
axes[2].set_ylabel("admissions")

plt.tight_layout()
plt.show()

In [ ]:
# === Feature Table ===
# this section is after we merged, on the feature table
# we have things added like age, history features, etc.

print("=" * 60)
print("FEATURE TABLE")
print("=" * 60)

snapshot = pd.read_csv(snapshot_path, low_memory=False)

# size after filtering to admissions with medications
print(f"snapshot rows (admissions kept): {len(snapshot):,}")
print(f"unique patients in snapshot: {snapshot['subject_id'].nunique():,}")

# Age
print("\nage at admission:")
age_desc = snapshot["age_at_admission"].describe().round(1)
print(age_desc.to_string())

# patient history, cold start admissions
print("\nnum_prior_admissions:")
prior_desc = snapshot["num_prior_admissions"].describe().round(2)
print(prior_desc.to_string())
first_visit = (snapshot["num_prior_admissions"] == 0).sum()
pct = first_visit / len(snapshot)
print(f"first-visit (cold-start) admissions: {first_visit:,} ({pct:.1%})")

# days between this and the most recent last admission
if "days_since_last_admission" in snapshot.columns:
    gap = snapshot["days_since_last_admission"].dropna()
    print("\ndays_since_last_admission (warm-start only):")
    gap_desc = gap.describe().round(1)
    print(gap_desc.to_string())

# Three plots: age dist, prior-admission count (log y because heavy
# tail), and the readmission gap distribution.
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(snapshot["age_at_admission"].dropna(), bins=40, edgecolor="white")
axes[0].set_title("age at admission")
axes[0].set_xlabel("age")
axes[0].set_ylabel("admissions")

axes[1].hist(snapshot["num_prior_admissions"], bins=40, edgecolor="white")
axes[1].set_title("num prior admissions")
axes[1].set_xlabel("prior admissions")
axes[1].set_ylabel("admissions")
axes[1].set_yscale("log")

if "days_since_last_admission" in snapshot.columns:
    axes[2].hist(
        snapshot["days_since_last_admission"].dropna(),
        bins=50,
        edgecolor="white",
    )
    axes[2].set_title("days since last admission")
    axes[2].set_xlabel("days")
    axes[2].set_ylabel("admissions")
    axes[2].set_yscale("log")

plt.tight_layout()
plt.show()

In [ ]:
# === Sparse Matrices ===
# this section is aggregated multi-hot features
# diagnoses, medication, labs, procedures
print("\n" + "=" * 60)
print("Sparse matrices")
print("=" * 60)

missing = []
for name, p in matrix_files.items():
    if not p.exists():
        missing.append(name)

if missing:
    print(f"missing matrices: {missing}. Run prepare.py first. Skipping section 3.")
else:
    for name, p in matrix_files.items():
        # print stats
        # is cur values, history values mostly populated or empty
        mat = load_npz(p)
        row_sums = mat.sum(axis=1)
        per_row = np.asarray(row_sums)
        per_row = per_row.flatten()

        total_cells = mat.shape[0] * mat.shape[1]
        density = mat.nnz / total_cells

        print(f"\n{name}: shape={mat.shape}, nnz={mat.nnz:,}, density={density:.4%}")
        mean_v = per_row.mean()
        med_v = np.median(per_row)
        max_v = per_row.max()
        print("  per-admission counts: mean=%.1f, median=%.0f, max=%d" % (mean_v, med_v, max_v))
        zero_rows = (per_row == 0).sum()
        print(f"  rows with zero entries: {zero_rows:,}")

In [ ]:
# === Interaction Table ===
print("\n" + "=" * 60)
print("Interaction Table")
print("=" * 60)

if not labels_path.exists():
    print(f"missing {labels_path}. Run prepare.py first. Skipping section 4.")
else:
    labels = pd.read_csv(labels_path, low_memory=False)
    n_labels = len(labels)
    n_pos = (labels["label"] == 1).sum()
    n_neg = (labels["label"] == 0).sum()
    n_drugs = labels["candidate_drug"].nunique()
    print("label rows: %d" % n_labels)
    print(f"positives: {n_pos:,}")
    print(f"negatives: {n_neg:,}")
    print(f"unique candidate drugs: {n_drugs:,}")

    # drugs per admission
    positives = labels[labels["label"] == 1]
    pos_per_admission = positives.groupby("hadm_id").size()
    print("\npositives per admission:")
    pos_desc = pos_per_admission.describe().round(2)
    print(pos_desc.to_string())

    # most popular drugs
    drug_popularity = positives["candidate_drug"].value_counts()
    print("\ntop 20 drugs by admissions administered:")
    top20_full = drug_popularity.head(20)
    print(top20_full.to_string())

    # what share of administrations are from the popular drugs
    # higher means stronger popularity bias, which means it may be harder to beat.
    n_unique_drugs = len(drug_popularity)
    top_10pct = int(n_unique_drugs * 0.1)
    if top_10pct < 1:
        top_10pct = 1
    top_share_sum = drug_popularity.head(top_10pct).sum()
    total_sum = drug_popularity.sum()
    share = top_share_sum / total_sum
    print(f"\ntop 10% of drugs ({top_10pct:,}) cover {share:.1%} of administrations")

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].hist(pos_per_admission, bins=50, edgecolor="white")
    axes[0].set_title("positives per admission")
    axes[0].set_xlabel("drugs administered")
    axes[0].set_ylabel("admissions")

    top20 = drug_popularity.head(20)
    axes[1].barh(top20.index[::-1], top20.values[::-1])
    axes[1].set_title("top 20 drugs")
    axes[1].set_xlabel("admissions")

    plt.tight_layout()
    plt.show()

## Training

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack, load_npz
from sklearn.metrics.pairwise import cosine_similarity

PROCESSED_DIR = Path("data/processed")
K = 20
KNN_NEIGHBORS = 50
BATCH_SIZE = 500
TEST_SIZE = 0.20
RANDOM_STATE = 42

In [ ]:
# split by patient, so that same patient doesn't show up in train and test
# this would cause leakage, so that all history stays on one side
def split_by_patient(features):
    rng = np.random.default_rng(RANDOM_STATE)
    subject_ids = features["subject_id"].drop_duplicates().to_numpy().copy()
    rng.shuffle(subject_ids)

    test_count = int(len(subject_ids) * TEST_SIZE)
    test_subjects = set(subject_ids[:test_count])
    train_subjects = set(subject_ids[test_count:])

    train_ids = features[features["subject_id"].isin(train_subjects)]["hadm_id"]
    test_ids = features[features["subject_id"].isin(test_subjects)]["hadm_id"]
    return set(train_ids), set(test_ids)

In [ ]:
# precision@k, recall@k, and ndcg@k
# we are predicting hadm_id -> ranked list of medications
# our test is hadm_id -> medications actually administered
def evaluate(predictions, ground_truth):
    precisions = []
    recalls = []
    ndcgs = []

    for hadm_id, true_drugs in ground_truth.items():
        preds = predictions.get(hadm_id, [])
        top_k = preds[:K]
        hits = set(top_k) & true_drugs

        # out of the K recommended drugs, how many were correct
        precisions.append(len(hits) / K)

        # out of the drugs actually given, how many did we include
        recalls.append(len(hits) / len(true_drugs))

        # rewards putting correct drugs higher
        ndcgs.append(ndcg_at_k(top_k, true_drugs))

    return np.mean(precisions), np.mean(recalls), np.mean(ndcgs)

In [ ]:
# ndcg@k for an admission
def ndcg_at_k(preds, true_drugs):
    dcg = 0.0
    rank = 1
    for drug in preds:
        if drug in true_drugs:
            dcg += 1 / np.log2(rank + 1)
        rank += 1

    # ideal is if our order is all right
    ideal_hits = min(len(true_drugs), K)
    ideal_dcg = 0.0
    for r in range(1, ideal_hits + 1):
        ideal_dcg += 1 / np.log2(r + 1)

    if ideal_dcg == 0:
        return 0.0
    return dcg / ideal_dcg

In [ ]:
# for the feature table
def build_dense_feature_matrix(features):
    drop_cols = ["subject_id", "hadm_id", "admittime", "edregtime", "edouttime"]
    dense = features.drop(columns=drop_cols, errors="ignore").copy()
    dense = pd.get_dummies(dense, dummy_na=True)
    dense = dense.fillna(0)
    arr = dense.to_numpy(dtype=np.float32)
    return csr_matrix(arr)

In [ ]:
# add in the dense matrices
def build_full_feature_matrix(features):
    current_dx = load_npz(PROCESSED_DIR / "current_dx_matrix.npz")
    prior_dx = load_npz(PROCESSED_DIR / "prior_dx_matrix.npz")
    prior_med = load_npz(PROCESSED_DIR / "prior_med_matrix.npz")
    dense_features = build_dense_feature_matrix(features)

    return hstack([current_dx, prior_dx, prior_med, dense_features], format="csr")

In [ ]:
features = pd.read_csv(
    PROCESSED_DIR / "patient_admission_snapshot.csv", low_memory=False
)

labels = pd.read_csv(PROCESSED_DIR / "admission_drug_labels.csv", low_memory=False)

# handle type
features["admittime"] = pd.to_datetime(features["admittime"])
features["hadm_id"] = features["hadm_id"].astype(int)
labels["hadm_id"] = labels["hadm_id"].astype(int)

positive_labels = labels[labels["label"] == 1]
interactions = positive_labels.rename(columns={"candidate_drug": "medication"})

train_ids, test_ids = split_by_patient(features)

train_interactions = interactions[interactions["hadm_id"].isin(train_ids)]
test_interactions = interactions[interactions["hadm_id"].isin(test_ids)]

ground_truth = {}
for hadm_id, grp in test_interactions.groupby("hadm_id"):
    ground_truth[hadm_id] = set(grp["medication"])

med_counts = train_interactions["medication"].value_counts()
top_meds = med_counts.index.tolist()

print("loaded %d admissions" % len(features))
print(f"train admissions: {len(train_ids):,}")
print(f"test admissions: {len(test_ids):,}")

In [ ]:
# model 1: overall medication popularity
# same list to every admission
popularity_preds = {}
for hadm_id in ground_truth:
    popularity_preds[hadm_id] = top_meds
p, r, n = evaluate(popularity_preds, ground_truth)
print(f"\npopularity -- precision@{K}: {p:.4f}, recall@{K}: {r:.4f}, ndcg@{K}: {n:.4f}")

In [ ]:
hadm_ids = features["hadm_id"].to_numpy()

train_rows_list = []
test_rows_list = []
for i in range(len(hadm_ids)):
    h = hadm_ids[i]
    if h in train_ids:
        train_rows_list.append(i)
    elif h in test_ids:
        test_rows_list.append(i)
train_rows = np.array(train_rows_list)
test_rows = np.array(test_rows_list)

train_hadm = hadm_ids[train_rows]
test_hadm = hadm_ids[test_rows]


In [ ]:
# Model 2: KNN using full admission feature set
print("Building full-feature KNN baseline...")
feature_matrix = build_full_feature_matrix(features)
train_features = feature_matrix[train_rows]
test_features = feature_matrix[test_rows]

train_drugs_by_hadm = {}
for hadm_id, grp in train_interactions.groupby("hadm_id"):
    train_drugs_by_hadm[hadm_id] = list(grp["medication"])

knn_preds = {}
for start in range(0, len(test_rows), BATCH_SIZE):
    batch = test_features[start : start + BATCH_SIZE]
    similarities = cosine_similarity(batch, train_features)

    for i, row_sims in enumerate(similarities):
        hadm_id = test_hadm[start + i]

        order_desc = np.argsort(row_sims)[::-1]
        neighbor_rows = order_desc[:KNN_NEIGHBORS]

        drug_scores = {}
        for neighbor_row in neighbor_rows:
            score = row_sims[neighbor_row]
            if score <= 0:
                continue

            neighbor_hadm_id = train_hadm[neighbor_row]
            if neighbor_hadm_id in train_drugs_by_hadm:
                neighbor_drugs = train_drugs_by_hadm[neighbor_hadm_id]
            else:
                neighbor_drugs = []
            for drug in neighbor_drugs:
                if drug in drug_scores:
                    drug_scores[drug] = drug_scores[drug] + score
                else:
                    drug_scores[drug] = score

        knn_preds[hadm_id] = sorted(drug_scores, key=drug_scores.get, reverse=True)

p, r, n = evaluate(knn_preds, ground_truth)
print(
    f"full-feature KNN -- precision@{K}: {p:.4f}, recall@{K}: {r:.4f}, ndcg@{K}: {n:.4f}"
)